In [1]:
# # OIBSIP — Data Analytics Track
# ## Task: Cleaning Data (Titanic Dataset)
# 
# **Objective:** Take the deliberately messy Titanic dataset and systematically transform it into a clean, analysis-ready dataset. Every cleaning decision is documented and justified below.
# 
# **Dataset:** Titanic Dataset (Kaggle — yasserh/titanic-dataset)
# **Author:** Sabbir
# **Track:** Data Analytics
# 

# ## Step 0 — Imports

import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)


# ## Step 1 — Load the Dataset and Initial Inspection

DATA_PATH = "/kaggle/input/datasets/yasserh/titanic-dataset/Titanic-Dataset.csv"

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
df.head()


df.info()


df.describe(include='all')


# ## Step 2 — Data Quality Report (BEFORE Cleaning)
# 
# Capturing null counts, duplicate counts, and dtypes now so we can build a
# "before vs. after" comparison at the end of the notebook.
# 

before_summary = {
    "row_count": len(df),
    "null_counts": df.isnull().sum(),
    "duplicate_rows": df.duplicated().sum(),
    "dtypes": df.dtypes
}

print("Row count:", before_summary["row_count"])
print("\nNull counts per column:\n", before_summary["null_counts"])
print("\nDuplicate rows:", before_summary["duplicate_rows"])
print("\nDtypes:\n", before_summary["dtypes"])


# **Observations:**
# - `Age` has a significant number of missing values.
# - `Cabin` is missing for the vast majority of rows.
# - `Embarked` has a small number of missing values.
# - `PassengerId`, `Survived`, `Pclass` are stored as integers but are really categorical/ID fields.
# - `Sex` and `Embarked` are text categories that need consistency checks.
# 

# ## Step 3 — Missing Data Handling
# 
# **Decisions and justification:**
# 
# - **`Age` (~20% missing):** Filled using the **median Age grouped by `Pclass` and `Sex`**, since age
#   distributions clearly differ by passenger class and gender on the Titanic — this is more accurate
#   than a single global median.
# - **`Embarked` (2 missing):** Filled with the **mode** (most frequent port), since only a couple of
#   rows are affected and mode imputation is standard practice for a categorical column with very few
#   missing values.
# - **`Cabin` (~77% missing):** Too sparse to impute reliably. Instead of dropping the column outright
#   (losing information), we convert it into a binary indicator column `Has_Cabin` (1 if cabin info was
#   recorded, 0 if not) — this preserves the fact that having a recorded cabin correlates with class/fare,
#   which is useful for analysis, and then drop the original noisy `Cabin` text column.
# 

# Age: median imputation grouped by Pclass and Sex
df['Age'] = df.groupby(['Pclass', 'Sex'])['Age'].transform(lambda x: x.fillna(x.median()))

# Fallback in case any group still has NaN (edge case safety)
df['Age'] = df['Age'].fillna(df['Age'].median())

# Embarked: mode imputation
embarked_mode = df['Embarked'].mode()[0]
df['Embarked'] = df['Embarked'].fillna(embarked_mode)

# Cabin: convert to Has_Cabin indicator, then drop original column
df['Has_Cabin'] = df['Cabin'].notna().astype(int)
df = df.drop(columns=['Cabin'])

print("Remaining nulls after handling:\n", df.isnull().sum())


# ## Step 4 — Duplicate Removal
# 
# Checking for fully duplicated rows and removing them, while documenting exactly how many were found.
# 

dupes_before = df.duplicated().sum()
df = df.drop_duplicates()
dupes_removed = dupes_before - df.duplicated().sum()

print(f"Duplicate rows found: {dupes_before}")
print(f"Duplicate rows removed: {dupes_removed}")
print(f"Row count after removing duplicates: {len(df)}")


# ## Step 5 — Standardisation
# 
# Making categorical text fields consistent (casing, whitespace) so they group and filter correctly.
# 

# Standardise text casing / whitespace
df['Sex'] = df['Sex'].str.strip().str.lower()
df['Embarked'] = df['Embarked'].str.strip().str.upper()

# Sanity check: unique values after standardisation
print("Sex values:", df['Sex'].unique())
print("Embarked values:", df['Embarked'].unique())


# ## Step 6 — Outlier Detection (IQR Method)
# 
# Applying the IQR method to the two continuous numeric columns: `Age` and `Fare`.
# 
# **Decision:**
# - **`Age`:** Outliers are biologically plausible (e.g., infants, elderly passengers were genuinely
#   on board) — so we **retain** them, only flagging for awareness.
# - **`Fare`:** A few extremely high fares are documented (first-class suites) but a handful of
#   outliers are far beyond the rest of the distribution and would distort analysis — so we **cap**
#   (winsorize) `Fare` at the upper IQR bound rather than deleting rows and losing passenger records.
# 

def iqr_bounds(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return lower, upper

# Age — detect only, no modification
age_lower, age_upper = iqr_bounds(df['Age'])
age_outliers = df[(df['Age'] < age_lower) | (df['Age'] > age_upper)]
print(f"Age outliers detected (kept as-is): {len(age_outliers)}")
print(f"Age IQR bounds: [{age_lower:.2f}, {age_upper:.2f}]")

# Fare — detect and cap
fare_lower, fare_upper = iqr_bounds(df['Fare'])
fare_outliers = df[(df['Fare'] < fare_lower) | (df['Fare'] > fare_upper)]
print(f"\nFare outliers detected: {len(fare_outliers)}")
print(f"Fare IQR bounds: [{fare_lower:.2f}, {fare_upper:.2f}]")

df['Fare'] = df['Fare'].clip(lower=max(fare_lower, 0), upper=fare_upper)
print("\nFare capped to upper IQR bound where needed.")


# ## Step 7 — Data Type Correction
# 
# Ensuring every column has the dtype that actually matches what it represents.
# 

df['PassengerId'] = df['PassengerId'].astype(str)
df['Survived']    = df['Survived'].astype('category')
df['Pclass']      = df['Pclass'].astype('category')
df['Sex']         = df['Sex'].astype('category')
df['Embarked']    = df['Embarked'].astype('category')
df['Has_Cabin']   = df['Has_Cabin'].astype('category')
df['Age']         = df['Age'].astype(float)
df['Fare']        = df['Fare'].astype(float)

df.dtypes


# ## Step 8 — Before vs. After Summary Table
# 

after_summary = {
    "row_count": len(df),
    "null_counts": df.isnull().sum().sum(),
    "duplicate_rows": df.duplicated().sum(),
}

summary_table = pd.DataFrame({
    "Metric": ["Row count", "Total null values", "Duplicate rows"],
    "Before Cleaning": [
        before_summary["row_count"],
        before_summary["null_counts"].sum(),
        before_summary["duplicate_rows"]
    ],
    "After Cleaning": [
        after_summary["row_count"],
        after_summary["null_counts"],
        after_summary["duplicate_rows"]
    ]
})

summary_table


# ## Step 9 — Save the Cleaned Dataset
# 

OUTPUT_PATH = "titanic_cleaned.csv"
df.to_csv(OUTPUT_PATH, index=False)
print(f"Cleaned dataset saved to: {OUTPUT_PATH}")
print(f"Final shape: {df.shape}")


# ## Conclusion
# 
# The raw Titanic dataset has been systematically cleaned:
# - Missing `Age` values imputed using group-wise medians (by `Pclass` and `Sex`).
# - Missing `Embarked` values imputed with the mode.
# - Sparse `Cabin` column converted into a usable `Has_Cabin` indicator and the noisy original dropped.
# - Duplicate rows checked and removed (count documented above).
# - Text categories standardised for consistent casing.
# - `Fare` outliers capped using the IQR method; `Age` outliers retained as biologically valid.
# - All columns corrected to appropriate dtypes.
# - Final cleaned dataset exported to `titanic_cleaned.csv`, ready for analysis.
#

Shape: (891, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB
Row count: 891

Null counts per column:
 PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare           